In [ ]:
%pip install ChemLogic

In [ ]:
import pandas as pd
from chemlogic.utils.Pipeline import Pipeline

# Load 50 molecules from the bundled dataset
df = pd.read_csv("../dataset.csv")
df = df.dropna(subset=["SMILES", "target"]).sample(50, random_state=42).reset_index(drop=True)

# Train a regression model
pipeline = Pipeline(
    dataset_name="checkpoint_demo",
    model_name="gnn",
    param_size=4,
    layers=1,
    smiles_list=df["SMILES"],
    labels=df["target"],
    task="regression",
)

train_loss, test_loss, r2, _ = pipeline.train_test_cycle(epochs=50)
(train_loss, test_loss, r2)

In [ ]:
# Save the trained model to a checkpoint
safetensors_path, json_path = pipeline.save_checkpoint(
    "../checkpoints/checkpoint_demo",
    metadata={"description": "regression demo, 50 molecules"},
)
print(f"Saved: {safetensors_path}")
print(f"       {json_path}")

In [ ]:
# Reload the checkpoint — no need to retrain
loaded = Pipeline.from_checkpoint(
    "../checkpoints/checkpoint_demo",
    smiles_list=df["SMILES"],
    labels=df["target"],
)

# Run inference on new molecules
test_smiles = ["CCO", "c1ccccc1", "CC(=O)O", "CN1CCC[C@H]1c2cccnc2"]
predictions = loaded.inference(test_smiles)

for smi, pred in zip(test_smiles, predictions):
    print(f"{smi:30s}  target = {pred:.4f}")

In [ ]:
# Multi-class example: bucket target into 3 activity classes
mc_labels = pd.cut(df["target"], bins=3, labels=[0, 1, 2]).astype(int).tolist()

mc_pipeline = Pipeline(
    dataset_name="multiclass_demo",
    model_name="gnn",
    param_size=4,
    layers=1,
    smiles_list=df["SMILES"],
    labels=mc_labels,
    task="multi_class",
    num_outputs=3,
)

train_loss, test_loss, auroc_ovr, _ = mc_pipeline.train_test_cycle(epochs=50)
(train_loss, test_loss, auroc_ovr)

In [ ]:
# Multi-regression: predict target and logP simultaneously
mr_labels = list(zip(df["target"].tolist(), df["logP"].tolist()))

mr_pipeline = Pipeline(
    dataset_name="multiregression_demo",
    model_name="gnn",
    param_size=4,
    layers=1,
    smiles_list=df["SMILES"],
    labels=mr_labels,
    task="multi_regression",
    num_outputs=2,
)

train_loss, test_loss, r2, _ = mr_pipeline.train_test_cycle(epochs=50)
(train_loss, test_loss, r2)